# FineWeb cached document lengths

**Hypothesis:** Qwen and GPT-2 document lengths are approximately linearly related. Large residuals or poor fit would instead indicate that tokenizer differences depend strongly on document content. Document lengths in the existing FineWeb cache have a long right tail. We expect the mean to exceed the median and the upper percentiles to be much larger than the median. A compact distribution or a sharp cutoff would instead suggest more uniform lengths or effects of cache-construction filtering.

**Scope and conclusions:** Scan every document in the selected cache, without additional filtering, sampling, shuffling, or truncation. Report document count, mean, population standard deviation (`ddof=0`), minimum, percentiles (1, 5, 25, 50, 75, 95, 99), and maximum in two units: cached GPT-2 tokens and Unicode characters (`len(text)`). Percentiles use NumPy's linear interpolation. These summaries describe this cache only; they do not establish lengths for all FineWeb or exact training truncation rates. The final cell measures Qwen lengths on a uniform random sample and uses a linear fit to approximate full-cache Qwen lengths. Construction filters may already have restricted the cache.

**Prerequisites:** Use the `stego` Conda environment with the repository dependencies, NumPy, Transformers, Jupyter, and an already completed FineWeb cache. The final cell needs the Qwen tokenizer, downloading its files into `$STEGO_ARTIFACTS_DIR/huggingface` if necessary; no model weights are loaded. Export `STEGO_ARTIFACTS_DIR` pointing to your existing artifacts directory before starting Jupyter. The default cache is `$STEGO_ARTIFACTS_DIR/datasets/fineweb/fineweb-500k`.

**Run:** From the repository root, run `conda activate stego`, then `PYTHONPATH="$PWD" jupyter lab ciphers/kirchenbauer_et_al/experiments/E20260916_fineweb_document_lengths.ipynb`. Select a kernel using `stego`, edit `CACHE_NAME` and the final cell's tokenizer, sample-size, seed, and batch-size constants if needed, and run all cells. The shared loader validates the completion marker, manifest, Parquet schemas, part sequence, and row counts. Missing or invalid caches raise an error; this notebook does not build or download the dataset cache.

**Outputs:** Statistics are printed below; no separate output files are written. Only integer lengths are retained for exact percentiles, so memory usage scales with the number of documents rather than their combined text size. The final cell additionally holds one batch of sampled text and token IDs in memory.

In [1]:
import os
from pathlib import Path

import numpy as np

from ciphers.kirchenbauer_et_al.src.cache_fineweb import load_fineweb_cache

CACHE_NAME = "fineweb-500k"

/opt/miniconda3/envs/stego/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The loader returns document dictionaries. This notebook consumes two required keys: `token_count`, the stored integer GPT-2 token count for the document, and `text`, the extracted document string whose Unicode character count is measured. Other cached fields are unused. The resulting `document_lengths` array has shape `(documents, 2)`, with GPT-2 token counts in column 0 and character counts in column 1; each document contributes once with equal weight.

In [2]:
cached_documents = load_fineweb_cache(CACHE_NAME, shuffle=False)
cache_directory = Path(os.environ["STEGO_ARTIFACTS_DIR"]) / "datasets" / "fineweb" / CACHE_NAME
print(f"Cache: {cache_directory}")

# Keep lengths only so the full corpus text need not fit in memory.
document_lengths = np.asarray(
    [(document["token_count"], len(document["text"])) for document in cached_documents],
    dtype=np.int64,
)
print(f"Documents: {len(document_lengths):,}")

Cache: /Users/4gate/git/StegoICMLMechInterp2026/artifacts/datasets/fineweb/fineweb-500k
Documents: 500,000


In [3]:
percentiles = (0, 1, 5, 25, 50, 75, 95, 99, 100)
statistic_labels = ("Minimum", "p01", "p05", "p25", "Median (p50)", "p75", "p95", "p99", "Maximum")

for column_index, unit in enumerate(("GPT-2 tokens (cached)", "Unicode characters")):
    lengths = document_lengths[:, column_index]
    print(f"\n{unit}")
    print(f"  {'Mean':<22} {lengths.mean():>14,.2f}")
    print(f"  {'Std. dev. (population)':<22} {lengths.std(ddof=0):>14,.2f}")
    for label, value in zip(statistic_labels, np.percentile(lengths, percentiles, method="linear"), strict=True):
        print(f"  {label:<22} {value:>14,.2f}")


GPT-2 tokens (cached)
  Mean                           693.49
  Std. dev. (population)       1,384.47
  Minimum                         33.00
  p01                             74.00
  p05                             99.00
  p25                            205.00
  Median (p50)                   404.00
  p75                            773.00
  p95                          2,000.00
  p99                          4,891.02
  Maximum                    130,040.00

Unicode characters
  Mean                         3,059.60
  Std. dev. (population)       5,979.06
  Minimum                        136.00
  p01                            301.00
  p05                            414.00
  p25                            892.00
  Median (p50)                 1,786.00
  p75                          3,450.00
  p95                          8,839.00
  p99                         21,469.00
  Maximum                    522,573.00


## Sampled Qwen lengths and a full-cache linear estimate

Uniformly select up to `16 * 1024` row indices **without replacement** from the entire cache (seed 42). Every document has inclusion probability `sample_size / cache_size`, even if the cache is sorted by length. Sort indices only after random selection to read efficiently; sorting does not change sample membership. Then rescan the same unshuffled cache and tokenize only those rows. Caches smaller than the cap use every document. `Qwen/Qwen3-4B-Base` matches the existing FineWeb inspection notebook. Count complete document token IDs with no special tokens, padding, or truncation; these lengths exclude control prefixes. The tokenizer output's required `input_ids` key is a list of token-ID lists in input order; each list's length is one document's Qwen count. The cache must remain unchanged between cells.

Fit ordinary least squares with an intercept: **Qwen tokens ≈ slope × cached GPT-2 tokens + intercept**. Print paired sample statistics, the ratio of sample token totals, in-sample R²/RMSE/MAE and residual standard deviation, then apply the fit to all cached GPT-2 lengths. Negative predictions are clipped to zero and counted. Report the share of the cache outside the sampled GPT-2 range to identify extrapolation. Fewer than two samples or constant GPT-2 lengths cannot identify a slope, so only observed summaries are printed in that case.

The table compares observed sample lengths for both tokenizers, exact full-cache GPT-2 lengths, and fitted full-cache Qwen lengths, including mean, population standard deviation, p10–p90, p95, and p99. **The sampled Qwen column is the direct empirical distribution estimate.** The fitted column describes predicted conditional means: it omits residual variation and can underestimate dispersion and misestimate tail percentiles. Fit diagnostics use the training sample, not independent validation; no confidence intervals or exact full-cache Qwen counts are claimed.

In [ ]:
from transformers import AutoTokenizer
# TODO(hadriano) this is not reviewed; we take AI here at its word; brief skim => probably OK?

QWEN_TOKENIZER_NAME = "Qwen/Qwen3-4B-Base"
MAX_SAMPLE_DOCUMENTS = 16 * 1024
SAMPLE_SEED = 42
TOKENIZATION_BATCH_SIZE = 32

sample_size = min(MAX_SAMPLE_DOCUMENTS, len(document_lengths))
# Sort after uniform selection; reading in cache order does not change sample membership.
sampled_indices = np.sort(np.random.default_rng(SAMPLE_SEED).choice(len(document_lengths), size=sample_size, replace=False))
qwen_tokenizer = AutoTokenizer.from_pretrained(
    QWEN_TOKENIZER_NAME,
    cache_dir=Path(os.environ["STEGO_ARTIFACTS_DIR"]) / "huggingface",
)
sampled_gpt2_counts = []
sampled_qwen_counts = []
text_batch = []
sample_position = 0
print(f"Tokenizing {sample_size:,} uniformly sampled documents with {QWEN_TOKENIZER_NAME} (seed {SAMPLE_SEED})")

for document_index, document in enumerate(load_fineweb_cache(CACHE_NAME, shuffle=False)):
    if document_index != sampled_indices[sample_position]:
        continue
    sampled_gpt2_counts.append(document["token_count"])
    text_batch.append(document["text"])
    sample_position += 1
    if len(text_batch) == TOKENIZATION_BATCH_SIZE or sample_position == sample_size:
        encoded_batch = qwen_tokenizer(text_batch, add_special_tokens=False, padding=False, truncation=False, return_attention_mask=False, return_token_type_ids=False)
        sampled_qwen_counts.extend(len(token_ids) for token_ids in encoded_batch["input_ids"])
        text_batch.clear()
        del encoded_batch
    if sample_position == sample_size:
        break

if sample_position != sample_size or not np.array_equal(sampled_gpt2_counts, document_lengths[sampled_indices, 0]):
    raise RuntimeError("Cache lengths changed between cells; rerun all cells against an unchanged cache.")
sampled_gpt2_lengths = np.asarray(sampled_gpt2_counts, dtype=np.float64)
sampled_qwen_lengths = np.asarray(sampled_qwen_counts, dtype=np.float64)
summary_columns = [
    ("Sample GPT-2", sampled_gpt2_lengths),
    ("Sample Qwen", sampled_qwen_lengths),
    ("Full GPT-2", document_lengths[:, 0]),
]
if sampled_gpt2_lengths.sum() > 0:
    print(f"Sample Qwen / GPT-2 token totals: {sampled_qwen_lengths.sum() / sampled_gpt2_lengths.sum():.4f}")

if sample_size < 2 or np.ptp(sampled_gpt2_lengths) == 0:
    print("Linear fit unavailable: need at least two distinct sampled GPT-2 lengths.")
else:
    centered_gpt2_lengths = sampled_gpt2_lengths - sampled_gpt2_lengths.mean()
    centered_qwen_lengths = sampled_qwen_lengths - sampled_qwen_lengths.mean()
    slope = np.dot(centered_gpt2_lengths, centered_qwen_lengths) / np.dot(centered_gpt2_lengths, centered_gpt2_lengths)
    intercept = sampled_qwen_lengths.mean() - slope * sampled_gpt2_lengths.mean()
    residuals = sampled_qwen_lengths - (slope * sampled_gpt2_lengths + intercept)
    residual_sum_squares = np.dot(residuals, residuals)
    total_sum_squares = np.dot(centered_qwen_lengths, centered_qwen_lengths)
    r_squared = 1 - residual_sum_squares / total_sum_squares if total_sum_squares > 0 else float("nan")
    print(f"OLS fit: Qwen tokens = {slope:.6f} * GPT-2 tokens + ({intercept:.3f})")
    print(f"In-sample R²: {r_squared:.6f} (undefined/NaN if Qwen lengths are constant)")
    print(f"In-sample RMSE: {np.sqrt(np.mean(residuals**2)):,.2f}; MAE: {np.mean(np.abs(residuals)):,.2f} Qwen tokens")
    print(f"Residual population standard deviation: {residuals.std(ddof=0):,.2f} Qwen tokens")
    predicted_qwen_lengths = slope * document_lengths[:, 0] + intercept
    print(f"Negative predictions clipped to zero: {np.count_nonzero(predicted_qwen_lengths < 0):,}")
    predicted_qwen_lengths = np.maximum(predicted_qwen_lengths, 0)
    outside_sample_range = (document_lengths[:, 0] < sampled_gpt2_lengths.min()) | (document_lengths[:, 0] > sampled_gpt2_lengths.max())
    print(f"Full-cache documents outside sampled GPT-2 range: {outside_sample_range.mean():.2%}")
    summary_columns.append(("Fitted full Qwen", predicted_qwen_lengths))

summary_percentiles = (0, 1, 5, 10, 20, 25, 30, 40, 50, 60, 70, 75, 80, 90, 95, 99, 100)
summary_labels = ["Documents", "Mean", "Std. dev. (population)"] + [f"p{percentile:02d}" for percentile in summary_percentiles]
summary_values = [[len(lengths), lengths.mean(), lengths.std(ddof=0), *np.percentile(lengths, summary_percentiles, method="linear")] for _, lengths in summary_columns]
print("\n" + f"{'Statistic':<24}" + "".join(f"{label:>20}" for label, _ in summary_columns))
for row_index, label in enumerate(summary_labels):
    print(f"{label:<24}" + "".join(f"{values[row_index]:>20,.2f}" for values in summary_values))
print("\nFitted full Qwen describes predicted means, omits residual variation, and approximates the full distribution.")

Tokenizing 16,384 uniformly sampled documents with Qwen/Qwen3-4B-Base (seed 42)
Sample Qwen / GPT-2 token totals: 0.9813
OLS fit: Qwen tokens = 0.991153 * GPT-2 tokens + (-6.778)
In-sample R²: 0.995774 (undefined/NaN if Qwen lengths are constant)
In-sample RMSE: 83.99; MAE: 27.05 Qwen tokens
Residual population standard deviation: 83.99 Qwen tokens
Negative predictions clipped to zero: 0
Full-cache documents outside sampled GPT-2 range: 0.01%

Statistic                       Sample GPT-2         Sample Qwen          Full GPT-2    Fitted full Qwen
Documents                          16,384.00           16,384.00          500,000.00          500,000.00
Mean                                  687.89              675.02              693.49              680.57
Std. dev. (population)              1,300.79            1,292.02            1,384.47            1,372.22
p00                                    33.00               31.00               33.00               25.93
p01                        